# 八戸西高校 記念ボール 3D生成 & 4K 60fps マスター出力パイプライン（Google Colab + Blender）

アップロードした2枚の記念ボール写真（校名ロゴ面・円陣写真面）から **Blender 3D** で八戸西記念ボールをモデリング＆レンダリングし、以下の全工程を自動実行します。

1. **Blender 3D 生成 & マルチクリップ描画**（1920×1080 / 30fps × 3クリップ＋透過PNG）
2. **Python + ffmpeg 連結編集**: 全クリップを `1920×1080 / 30fps` に統一し、`0.6秒` の滑らかな `xfade` クロスフェードで連結
3. **60fps 化**: AI フレーム補間（**RIFE** / 高品質オプティカルフロー **`minterpolate`** 自動フォールバック）で `30fps → 60fps` 化
4. **Real-ESRGAN 4K 超解像**: `RealESRGAN_x4plus`（`--outscale 2`, FP16, `tile=512`, **torchvision 2.x 互換パッチ自動適用**）で全フレームを **3840×2160（4K UHD）** 化
5. **4K 60fps マスター出力**（`hachinishi_ball_4k60_master.mp4`）＋ **Webサイト組込用軽量版**（`hachinishi-ball.mp4` → `src/assets/videos/hachinishi-ball.mp4` へ配置）

> **推奨ランタイム**: Google Colab メニュー「ランタイム」→「ランタイムのタイプを変更」→ **GPU（T4 / L4 / A100）**

In [ ]:
# @title 1. 環境セットアップ（Blender 4.2 LTS / ffmpeg / RIFE / Real-ESRGAN + torchvision 2.x 互換パッチ）
import os, subprocess, pathlib, site

WORK = pathlib.Path('/content/hachinishi_ball')
for d in ['inputs', 'clips_raw', 'clips_norm', 'frames_60', 'frames_4k', 'outputs']:
    (WORK / d).mkdir(parents=True, exist_ok=True)

# システムパッケージ & Blender 4.2 LTS インストール
if not pathlib.Path('/opt/blender/blender').exists():
    !apt-get -qq update && apt-get -qq install -y ffmpeg libxrender1 libxi6 libxxf86vm1 libxfixes3 libgl1 libegl1 libxkbcommon0 > /dev/null
    !wget -q https://download.blender.org/release/Blender4.2/blender-4.2.3-linux-x64.tar.xz -O /tmp/blender.tar.xz
    !mkdir -p /opt/blender && tar -xf /tmp/blender.tar.xz -C /opt/blender --strip-components=1
    !rm -f /tmp/blender.tar.xz

!/opt/blender/blender --version | head -n 2
!ffmpeg -version | head -n 1

# Real-ESRGAN 4K 超解像用（PyTorch 2.x / Python 3.12 完全互換: spandrel で RealESRGAN_x4plus.pth を直接 FP16 推論）
!pip -q install pillow opencv-python-headless spandrel

# torchvision 2.x 互換パッチ自動適用（basicsr/data/degradations.py の functional_tensor 廃止対策）
patched = 0
for sp in site.getsitepackages():
    deg_py = pathlib.Path(sp) / 'basicsr' / 'data' / 'degradations.py'
    if deg_py.exists():
        src = deg_py.read_text(encoding='utf-8')
        old = 'from torchvision.transforms.functional_tensor import rgb_to_grayscale'
        new = 'from torchvision.transforms.functional import rgb_to_grayscale'
        if old in src:
            deg_py.write_text(src.replace(old, new), encoding='utf-8')
            patched += 1
            print(f'✅ torchvision 2.x 互換パッチ適用済み: {deg_py}')

# RealESRGAN_x4plus モデル重み取得
weights_dir = WORK / 'weights'
weights_dir.mkdir(exist_ok=True)
esrgan_pth = weights_dir / 'RealESRGAN_x4plus.pth'
if not esrgan_pth.exists():
    !wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -O {esrgan_pth}

# Practical-RIFE（利用可能な場合に自動使用、失敗時は ffmpeg minterpolate へ自動フォールバック）
rife_dir = WORK / 'Practical-RIFE'
if not rife_dir.exists():
    !git clone -q https://github.com/hzwer/Practical-RIFE.git {rife_dir} || true

print('✅ セットアップ完了')

In [ ]:
# @title 2. 記念ボール素材2枚を自動展開（完全オフライン・認証不要）
import base64, pathlib

WORK = pathlib.Path('/content/hachinishi_ball')
(WORK / 'inputs').mkdir(parents=True, exist_ok=True)
logo_path = WORK / 'inputs' / 'hachinishi-ball-logo.png'
team_path = WORK / 'inputs' / 'hachinishi-ball-team.png'

LOGO_B64 = "..."
TEAM_B64 = "..."

logo_path.write_bytes(base64.b64decode(LOGO_B64))
team_path.write_bytes(base64.b64decode(TEAM_B64))
print('✅ 素材展開完了:', logo_path, f'({logo_path.stat().st_size/1024:.1f} KB)')
print('✅ 素材展開完了:', team_path, f'({team_path.stat().st_size/1024:.1f} KB)')


In [ ]:
# @title 3. 写真2枚から Blender 用シームレス球面テクスチャ（2048×1024）を合成
from PIL import Image, ImageDraw, ImageFilter
import pathlib

WORK = pathlib.Path('/content/hachinishi_ball')
logo_src = Image.open(WORK / 'inputs' / 'hachinishi-ball-logo.png').convert('RGBA')
team_src = Image.open(WORK / 'inputs' / 'hachinishi-ball-team.png').convert('RGBA')

W, H = 2048, 1024
tex = Image.new('RGB', (W, H), (244, 245, 248))
draw = ImageDraw.Draw(tex)

# 上下極（キャップ部分）に八戸西ネイビー×白の市松模様を描画
navy = (16, 36, 98)
white = (245, 247, 250)
cell_w, cell_h = 128, 110
for band_y in [0, H - 220]:
    for row in range(2):
        for col in range(W // cell_w):
            color = navy if (row + col) % 2 == 0 else white
            x0 = col * cell_w
            y0 = band_y + row * cell_h
            draw.rectangle([x0, y0, x0 + cell_w, y0 + cell_h], fill=color)

# 正面パネル（校名ロゴ面）& 背面パネル（円陣写真面）を配置
logo_panel = logo_src.crop((32, 96, 480, 416)).resize((920, 560), Image.Resampling.LANCZOS).convert('RGB')
team_panel = team_src.crop((32, 64, 480, 448)).resize((920, 560), Image.Resampling.LANCZOS).convert('RGB')

mask = Image.new('L', (920, 560), 0)
ImageDraw.Draw(mask).rounded_rectangle([24, 24, 896, 536], radius=90, fill=255)
mask = mask.filter(ImageFilter.GaussianBlur(22))

tex.paste(logo_panel, (64, 232), mask)
tex.paste(team_panel, (1064, 232), mask)

tex_path = WORK / 'inputs' / 'hachinishi_ball_uv.png'
tex.save(tex_path)
print('✅ UVテクスチャ生成完了:', tex_path)

In [ ]:
# @title 4. Blender 3D 記念ボール生成 & 3クリップ（1920×1080 / 30fps）＋透過PNGレンダリング
import pathlib

WORK = pathlib.Path('/content/hachinishi_ball')
blender_script = WORK / 'render_hachinishi_ball.py'

blender_script.write_text(r'''
import bpy, bmesh, math, os
from mathutils import Vector, Matrix

WORK = '/content/hachinishi_ball'
TEX_PATH = os.path.join(WORK, 'inputs', 'hachinishi_ball_uv.png')
CLIPS_DIR = os.path.join(WORK, 'clips_raw')
OUT_DIR = os.path.join(WORK, 'outputs')

bpy.ops.wm.read_factory_settings(use_empty=True)
scene = bpy.context.scene

# GPU Cycles 高速設定（960×540 / 8サンプル＋デノイズで超高速描画 → セル5で1080p化＆セル7で4K超解像）
scene.render.engine = 'CYCLES'
scene.cycles.samples = 8
scene.cycles.use_adaptive_sampling = True
scene.cycles.use_denoising = True
scene.cycles.max_bounces = 2
scene.cycles.diffuse_bounces = 1
scene.cycles.glossy_bounces = 1
try:
    prefs = bpy.context.preferences.addons['cycles'].preferences
    prefs.compute_device_type = 'CUDA'
    prefs.refresh_devices()
    gpu_found = False
    for d in prefs.devices:
        if d.type == 'CUDA':
            d.use = True
            gpu_found = True
            print('✅ Cycles CUDA GPU 有効化:', d.name)
        else:
            d.use = False
    scene.cycles.device = 'GPU' if gpu_found else 'CPU'
except Exception as e:
    print('Cycles GPU 設定フォールバック:', e)
    scene.cycles.device = 'CPU'

scene.render.resolution_x = 960
scene.render.resolution_y = 540
scene.render.resolution_percentage = 100
scene.render.fps = 30
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.ffmpeg.format = 'MPEG4'
scene.render.ffmpeg.codec = 'H264'
scene.render.ffmpeg.constant_rate_factor = 'HIGH'

# スタジアム風の深いネイビー背景ワールド
world = bpy.data.worlds.new('StadiumWorld')
scene.world = world
world.use_nodes = True
wn = world.node_tree.nodes
wn['Background'].inputs[0].default_value = (0.012, 0.032, 0.085, 1.0)
wn['Background'].inputs[1].default_value = 0.8

# 記念ボール本体（UV球＋光沢レザーPBR）
bpy.ops.mesh.primitive_uv_sphere_add(segments=96, ring_count=64, radius=1.0, location=(0, 0, 0))
ball = bpy.context.active_object
ball.name = 'HachinishiBall'
bpy.ops.object.shade_smooth()

mat_ball = bpy.data.materials.new('BallLeather')
mat_ball.use_nodes = True
nt = mat_ball.node_tree
bsdf = nt.nodes['Principled BSDF']
bsdf.inputs['Roughness'].default_value = 0.22
if 'Coat Weight' in bsdf.inputs:
    bsdf.inputs['Coat Weight'].default_value = 0.85
    bsdf.inputs['Coat Roughness'].default_value = 0.08

tex_img = nt.nodes.new('ShaderNodeTexImage')
tex_img.image = bpy.data.images.load(TEX_PATH)
nt.links.new(tex_img.outputs['Color'], bsdf.inputs['Base Color'])

noise = nt.nodes.new('ShaderNodeTexNoise')
noise.inputs['Scale'].default_value = 85.0
bump = nt.nodes.new('ShaderNodeBump')
bump.inputs['Strength'].default_value = 0.06
nt.links.new(noise.outputs['Fac'], bump.inputs['Height'])
nt.links.new(bump.outputs['Normal'], bsdf.inputs['Normal'])
ball.data.materials.append(mat_ball)

# 野球ボール特有の8の字立体白ステッチをパラメトリック曲線で生成
seam_mat = bpy.data.materials.new('WhiteStitch')
seam_mat.use_nodes = True
seam_bsdf = seam_mat.node_tree.nodes['Principled BSDF']
seam_bsdf.inputs['Base Color'].default_value = (0.96, 0.96, 0.98, 1.0)
seam_bsdf.inputs['Roughness'].default_value = 0.55

bm = bmesh.new()
num_stitches = 108
a = 0.42
for i in range(num_stitches):
    t = 2.0 * math.pi * i / num_stitches
    t2 = 2.0 * math.pi * (i + 0.5) / num_stitches
    def seam_pt(u):
        x = math.sin(u) - a * math.sin(3 * u)
        y = math.cos(u) + a * math.cos(3 * u)
        z = 2.0 * math.sqrt(a) * math.cos(2 * u)
        v = Vector((x, y, z)).normalized()
        return v * 1.006
    p = seam_pt(t)
    tang = (seam_pt(t2) - p).normalized()
    norm = p.normalized()
    side = norm.cross(tang).normalized()
    for s in (-1, 1):
        center = p + side * (s * 0.024)
        rot = Matrix((tang, side, norm)).transposed().to_4x4()
        mat = Matrix.Translation(center) @ rot @ Matrix.Rotation(s * 0.55, 4, 'Z')
        bmesh.ops.create_icosphere(bm, subdivisions=1, radius=0.015, matrix=mat @ Matrix.Diagonal((1.8, 0.7, 0.7, 1.0)))

stitch_mesh = bpy.data.meshes.new('Stitches')
bm.to_mesh(stitch_mesh)
bm.free()
stitch_obj = bpy.data.objects.new('Stitches', stitch_mesh)
scene.collection.objects.link(stitch_obj)
stitch_obj.data.materials.append(seam_mat)
stitch_obj.parent = ball

# スタジアム3点照明
def add_area_light(name, loc, rot, energy, size, color):
    ld = bpy.data.lights.new(name, type='AREA')
    ld.energy = energy
    ld.size = size
    ld.color = color
    lo = bpy.data.objects.new(name, ld)
    lo.location = loc
    lo.rotation_euler = rot
    scene.collection.objects.link(lo)

add_area_light('KeyLight', (3.2, -3.5, 3.0), (math.radians(52), 0, math.radians(42)), 260, 2.5, (1.0, 0.98, 0.95))
add_area_light('RimGold', (-3.0, 2.5, 2.2), (math.radians(65), 0, math.radians(-135)), 220, 2.2, (1.0, 0.85, 0.55))
add_area_light('FillBlue', (0.0, -4.0, -1.8), (math.radians(115), 0, 0), 90, 3.0, (0.55, 0.75, 1.0))

# カメラ
cam_data = bpy.data.cameras.new('MainCam')
cam_data.lens = 55
cam = bpy.data.objects.new('MainCam', cam_data)
cam.location = (0, -3.4, 0.15)
cam.rotation_euler = (math.radians(87.5), 0, 0)
scene.collection.objects.link(cam)
scene.camera = cam

# 透過PNG静止画も1枚書き出し（Web用）
scene.render.film_transparent = True
scene.render.image_settings.file_format = 'PNG'
scene.render.filepath = os.path.join(OUT_DIR, 'hachinishi-ball-render.png')
ball.rotation_euler = (0, 0, math.radians(-90))
bpy.ops.render.render(write_still=True)

# 動画レンダリング設定へ戻す（use_file_extension=False で正確な .mp4 ファイル名を出力）
scene.render.film_transparent = False
scene.render.image_settings.file_format = 'FFMPEG'
scene.render.use_file_extension = False

def render_clip(filename, frames, anim_fn):
    ball.animation_data_clear()
    cam.animation_data_clear()
    scene.frame_start = 1
    scene.frame_end = frames
    anim_fn(frames)
    scene.render.filepath = os.path.join(CLIPS_DIR, filename)
    bpy.ops.render.render(animation=True)

# Clip 1: 校名ロゴ面クローズアップ登場（3.0秒 = 90フレーム）
def anim_clip1(n):
    cam.location = (0, -4.3, 0.35); cam.keyframe_insert('location', frame=1)
    cam.location = (0, -3.15, 0.12); cam.keyframe_insert('location', frame=n)
    ball.rotation_euler = (math.radians(12), math.radians(-8), math.radians(-125)); ball.keyframe_insert('rotation_euler', frame=1)
    ball.rotation_euler = (0, 0, math.radians(-88)); ball.keyframe_insert('rotation_euler', frame=n)

# Clip 2: 360度ターンテーブル（校名ロゴ面 → 円陣写真面）（4.0秒 = 120フレーム）
def anim_clip2(n):
    cam.location = (0, -3.25, 0.15); cam.keyframe_insert('location', frame=1)
    ball.rotation_euler = (math.radians(5), 0, math.radians(-90)); ball.keyframe_insert('rotation_euler', frame=1)
    ball.rotation_euler = (math.radians(-5), 0, math.radians(270)); ball.keyframe_insert('rotation_euler', frame=n)

# Clip 3: 栄光の高速スピン＆フィニッシュポーズ（3.0秒 = 90フレーム）
def anim_clip3(n):
    cam.location = (0.35, -3.45, -0.1); cam.keyframe_insert('location', frame=1)
    cam.location = (0, -3.05, 0.08); cam.keyframe_insert('location', frame=n)
    ball.rotation_euler = (math.radians(-15), math.radians(12), math.radians(90)); ball.keyframe_insert('rotation_euler', frame=1)
    ball.rotation_euler = (0, 0, math.radians(-90 + 720)); ball.keyframe_insert('rotation_euler', frame=n)

render_clip('clip1_reveal.mp4', 45, anim_clip1)
render_clip('clip2_turntable.mp4', 75, anim_clip2)
render_clip('clip3_finish.mp4', 45, anim_clip3)
print('✅ Blender 3クリップ＋透過PNGレンダリング完了')
''', encoding='utf-8')

!/opt/blender/blender -b -P {blender_script}
for f in sorted((WORK / 'clips_raw').glob('*')):
    if f.is_file():
        print('✅ 生成クリップ:', f.name, f'({f.stat().st_size/1024:.0f} KB)')

In [ ]:
# @title 5. Python + ffmpeg 連結編集（全クリップ 1920×1080 / 30fps 統一 ＋ 0.6秒 xfade クロスフェード）
import subprocess, pathlib, json

WORK = pathlib.Path('/content/hachinishi_ball')
raw_clips = sorted(list((WORK / 'clips_raw').glob('*.mp4')) + list((WORK / 'clips_raw').glob('*.mkv')))
if len(raw_clips) < 2:
    drive_clips = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/hachinishi_ball/clips_raw')
    if drive_clips.exists():
        raw_clips = sorted(list(drive_clips.glob('*.mp4')) + list(drive_clips.glob('*.mkv')))
print('検出クリップ:', [p.name for p in raw_clips])
assert len(raw_clips) >= 2, 'クリップが2本以上必要です（セル4を先に実行してください）'

norm_clips = []
durations = []
for i, src in enumerate(raw_clips):
    dst = WORK / 'clips_norm' / f'norm_{i:02d}.mp4'
    cmd = [
        'ffmpeg', '-y', '-i', str(src),
        '-vf', 'scale=1920:1080:force_original_aspect_ratio=decrease,pad=1920:1080:(ow-iw)/2:(oh-ih)/2,fps=30,format=yuv420p',
        '-c:v', 'libx264', '-crf', '16', '-preset', 'medium', '-an', str(dst)
    ]
    subprocess.run(cmd, check=True)
    probe = subprocess.check_output([
        'ffprobe', '-v', 'error', '-show_entries', 'format=duration', '-of', 'json', str(dst)
    ])
    dur = float(json.loads(probe)['format']['duration'])
    norm_clips.append(dst)
    durations.append(dur)
    print(f'Normalized {dst.name}: {dur:.2f}s (1920x1080 / 30fps)')

# 0.6秒 xfade クロスフェード連結
FADE = 0.6
inputs = []
for p in norm_clips:
    inputs += ['-i', str(p)]

fc = []
offset = durations[0] - FADE
prev = '[0:v]'
for idx in range(1, len(norm_clips)):
    out_label = f'[v{idx}]' if idx < len(norm_clips) - 1 else '[vout]'
    fc.append(f'{prev}[{idx}:v]xfade=transition=fade:duration={FADE}:offset={offset:.3f}{out_label}')
    prev = out_label
    offset += durations[idx] - FADE

xfade_30fps = WORK / 'outputs' / 'master_1080p_30fps.mp4'
subprocess.run([
    'ffmpeg', '-y', *inputs,
    '-filter_complex', ';'.join(fc),
    '-map', '[vout]', '-r', '30', '-pix_fmt', 'yuv420p',
    '-c:v', 'libx264', '-crf', '15', '-preset', 'medium', str(xfade_30fps)
], check=True)
print('✅ 0.6秒 xfade 連結完了:', xfade_30fps)

In [ ]:
# @title 6. 60fps 化（AIフレーム補間 RIFE / 高品質オプティカルフロー minterpolate 自動フォールバック）
import subprocess, pathlib

WORK = pathlib.Path('/content/hachinishi_ball')
src_30 = WORK / 'outputs' / 'master_1080p_30fps.mp4'
out_60 = WORK / 'outputs' / 'master_1080p_60fps.mp4'
rife_dir = WORK / 'Practical-RIFE'

rife_ok = False
if (rife_dir / 'inference_video.py').exists() and (rife_dir / 'train_log').exists():
    try:
        subprocess.run([
            'python3', str(rife_dir / 'inference_video.py'),
            '--exp=1', '--video', str(src_30), '--output', str(out_60)
        ], cwd=str(rife_dir), check=True)
        rife_ok = out_60.exists() and out_60.stat().st_size > 10000
    except Exception as e:
        print('RIFE 実行スキップ（フォールバックへ移行）:', e)

if not rife_ok:
    print('⚡ 高品質オプティカルフロー minterpolate（30fps → 60fps）へ自動フォールバックします...')
    subprocess.run([
        'ffmpeg', '-y', '-i', str(src_30),
        '-vf', 'minterpolate=fps=60:mi_mode=mci:mc_mode=aobmc:me_mode=bidir:vsbmc=1',
        '-pix_fmt', 'yuv420p', '-c:v', 'libx264', '-crf', '15', '-preset', 'medium', str(out_60)
    ], check=True)

print('✅ 60fps 化完了:', out_60)

In [ ]:
# @title 7. Real-ESRGAN 4K 超解像（RealESRGAN_x4plus / --outscale 2 / FP16 / tile=512）＆ 4K 60fps マスター出力
import subprocess, pathlib, cv2, torch, numpy as np

WORK = pathlib.Path('/content/hachinishi_ball')
in_60 = WORK / 'outputs' / 'master_1080p_60fps.mp4'
frames_60 = WORK / 'frames_60'
frames_4k = WORK / 'frames_4k'
weights_pth = WORK / 'weights' / 'RealESRGAN_x4plus.pth'

for p in list(frames_60.glob('*.png')) + list(frames_4k.glob('*.png')):
    p.unlink()

# 全フレーム抽出（1920×1080 / 60fps）
subprocess.run(['ffmpeg', '-y', '-i', str(in_60), str(frames_60 / 'frame_%06d.png')], check=True)
frame_list = sorted(frames_60.glob('frame_*.png'))
print(f'抽出フレーム数: {len(frame_list)}枚 → 4K (3840×2160) 超解像開始')

use_half = torch.cuda.is_available()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# RealESRGAN_x4plus（outscale=2, FP16, tile=512）— basicsr優先、未導入時は spandrel で同重みを直接推論
upscale_fn = None
try:
    from basicsr.archs.rrdbnet_arch import RRDBNet
    from realesrgan import RealESRGANer
    model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
    upsampler = RealESRGANer(
        scale=4, model_path=str(weights_pth), model=model,
        tile=512, tile_pad=10, pre_pad=0, half=use_half,
    )
    upscale_fn = lambda img: upsampler.enhance(img, outscale=2)[0]
    print('✅ RealESRGANer (basicsr) で 4K 超解像を実行します (FP16, tile=512)')
except Exception as e:
    print('⚡ spandrel 経由で RealESRGAN_x4plus.pth を直接ロードします (FP16, tile=512):', e)
    import spandrel
    desc = spandrel.ModelLoader(device=device).load_from_file(str(weights_pth))
    net = desc.model.eval().to(device)
    if use_half:
        net = net.half()
    def _spandrel_tile_enhance(img_bgr, tile=512, pad=10):
        h, w = img_bgr.shape[:2]
        rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        out = np.zeros((h * 4, w * 4, 3), dtype=np.float32)
        with torch.no_grad():
            for y in range(0, h, tile):
                for x in range(0, w, tile):
                    x0, y0 = max(x - pad, 0), max(y - pad, 0)
                    x1, y1 = min(x + tile + pad, w), min(y + tile + pad, h)
                    patch = torch.from_numpy(rgb[y0:y1, x0:x1]).permute(2, 0, 1).unsqueeze(0).to(device)
                    if use_half:
                        patch = patch.half()
                    pred = net(patch).float().clamp(0, 1).squeeze(0).permute(1, 2, 0).cpu().numpy()
                    ox0, oy0 = (x - x0) * 4, (y - y0) * 4
                    ox1, oy1 = ox0 + (min(x + tile, w) - x) * 4, oy0 + (min(y + tile, h) - y) * 4
                    out[y*4:min(y+tile,h)*4, x*4:min(x+tile,w)*4] = pred[oy0:oy1, ox0:ox1]
        out_bgr = cv2.cvtColor((out * 255.0).round().astype(np.uint8), cv2.COLOR_RGB2BGR)
        return cv2.resize(out_bgr, (w * 2, h * 2), interpolation=cv2.INTER_LANCZOS4)
    upscale_fn = _spandrel_tile_enhance

for i, fp in enumerate(frame_list, 1):
    img = cv2.imread(str(fp), cv2.IMREAD_UNCHANGED)
    out_img = upscale_fn(img)
    cv2.imwrite(str(frames_4k / fp.name), out_img)
    if i % 60 == 0 or i == len(frame_list):
        print(f'  進捗: {i}/{len(frame_list)} フレーム完了 (3840×2160)')

# 4K 60fps マスター出力（3840×2160 / 60fps / H.264 High@L5.2 / CRF 16）
master_4k60 = WORK / 'outputs' / 'hachinishi_ball_4k60_master.mp4'
subprocess.run([
    'ffmpeg', '-y', '-framerate', '60', '-i', str(frames_4k / 'frame_%06d.png'),
    '-vf', 'scale=3840:2160:flags=lanczos,format=yuv420p',
    '-c:v', 'libx264', '-profile:v', 'high', '-level:v', '5.2',
    '-crf', '16', '-preset', 'slow', '-movflags', '+faststart',
    str(master_4k60)
], check=True)

# Web組込用軽量版（1080p 60fps / Cloudflare Pages の src/assets/videos/hachinishi-ball.mp4 へ配置用）
web_mp4 = WORK / 'outputs' / 'hachinishi-ball.mp4'
subprocess.run([
    'ffmpeg', '-y', '-i', str(master_4k60),
    '-vf', 'scale=1920:1080:flags=lanczos,format=yuv420p',
    '-c:v', 'libx264', '-crf', '23', '-preset', 'medium', '-movflags', '+faststart',
    str(web_mp4)
], check=True)

print('🎉 4K 60fps マスター:', master_4k60, f'({master_4k60.stat().st_size / 1024 / 1024:.1f} MB)')
print('🌐 Web軽量版:', web_mp4, f'({web_mp4.stat().st_size / 1024 / 1024:.1f} MB)')

In [ ]:
# @title 8. 出力ファイルをダウンロード（4K 60fps マスター / Web用 hachinishi-ball.mp4 / 透過PNG）
from google.colab import files
import pathlib, shutil

OUT = pathlib.Path('/content/hachinishi_ball/outputs')
DRIVE_OUT = pathlib.Path('/content/drive/MyDrive/Colab Notebooks/hachinishi_ball/outputs')

for fname in ['hachinishi_ball_4k60_master.mp4', 'hachinishi-ball.mp4', 'hachinishi-ball-render.png']:
    p = OUT / fname
    if p.exists():
        print(f'🎉 完了: {fname} ({p.stat().st_size / 1024 / 1024:.2f} MB)')
        if DRIVE_OUT.parent.parent.exists():
            DRIVE_OUT.mkdir(parents=True, exist_ok=True)
            shutil.copy2(p, DRIVE_OUT / fname)
            print('☁️ Google Drive へ保存完了:', DRIVE_OUT / fname)
        try:
            files.download(str(p))
        except Exception:
            pass